In [1]:
import requests

In [2]:
def get_weather_description(code):
    weather_codes = {
        0: "Clear sky",
        1: "Mainly clear",
        2: "Partly cloudy",
        3: "Overcast",
        45: "Fog",
        48: "Depositing rime fog",
        51: "Light drizzle",
        53: "Moderate drizzle",
        55: "Dense drizzle",
        61: "Slight rain",
        63: "Moderate rain",
        65: "Heavy rain",
        71: "Slight snowfall",
        73: "Moderate snowfall",
        75: "Heavy snowfall",
        80: "Rain showers",
        81: "Moderate rain showers",
        82: "Heavy rain showers",
        95: "Thunderstorm"
    }
    return weather_codes.get(code, "Unknown weather condition")


try:
    # Step 1: Get city name
    city = input("Enter city name: ").strip()

    if not city:
        raise ValueError("City name cannot be empty.")

    # Step 2: Get city coordinates
    geo_url = "https://geocoding-api.open-meteo.com/v1/search"

    geo_response = requests.get(
        geo_url,
        params={"name": city, "count": 1, "language": "en"},
        timeout=15
    )
    geo_response.raise_for_status()

    geo_data = geo_response.json()

    if not geo_data.get("results"):
        print("City not found.")
    else:
        location = geo_data["results"][0]
        latitude = location["latitude"]
        longitude = location["longitude"]

        # Step 3: Fetch current weather
        weather_url = "https://api.open-meteo.com/v1/forecast"

        weather_response = requests.get(
            weather_url,
            params={
                "latitude": latitude,
                "longitude": longitude,
                "current": "temperature_2m,wind_speed_10m,weather_code"
            },
            timeout=15
        )
        weather_response.raise_for_status()

        weather_data = weather_response.json()
        current = weather_data["current"]

        # Step 4: Display weather report
        print("\n--- LIVE WEATHER REPORT ---")
        print("City:", location["name"])
        print("Country:", location.get("country", "Unknown"))
        print("Temperature:",
              current["temperature_2m"], "°C")
        print("Wind Speed:",
              current["wind_speed_10m"], "km/h")
        print("Weather Description:",
              get_weather_description(current["weather_code"]))
        print("Weather Code:", current["weather_code"])
        print("Observation Time:", current["time"])

except requests.RequestException as error:
    print("Unable to fetch weather data:", error)
except (KeyError, ValueError) as error:
    print("Error:", error)


Enter city name:  pune



--- LIVE WEATHER REPORT ---
City: Pune
Country: India
Temperature: 30.3 °C
Wind Speed: 6.5 km/h
Weather Description: Overcast
Weather Code: 3
Observation Time: 2026-10-24T05:00
